# 实验数据划分
从APR数据集中划分出白盒设置下的query_set.jsonl、黑盒设置下的query_set.jsonl和APR知识库rag_data.jsonl

## 1 依赖导入与全局配置

In [ ]:
import os
import sys
import json
import random
from pathlib import Path


# =========================================================
# 超参数
# =========================================================
CORPUS_SIZE   = 50000   # 语料库（RAG 知识库）规模
# QUERY_SIZE    = 25000   # 查询集规模（白盒 / 黑盒各自的最终大小）
QUERY_SIZE    = 12500   # 查询集规模（白盒 / 黑盒各自的最终大小）
# OVERLAP_RATE  = 1     # 查询集与语料库的数据重叠率 θ
OVERLAP_RATE  = 0.1     # 查询集与语料库的数据重叠率 θ
SEED          = 42      # 随机种子

# =========================================================
# 文件路径
# =========================================================
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR        = os.path.join(PROJECT_ROOT, 'data', 'raw', 'bfp')
CORPUS_OUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'corpus', 'bfp')
QUERY_OUT_DIR  = os.path.join(PROJECT_ROOT, 'data', 'query')

COCONUT_PATH   = os.path.join(RAW_DIR, 'CoCoNut.jsonl')
CODEFLAWS_PATH = os.path.join(RAW_DIR, 'Codeflaws.json')
DEEPFIX_PATH   = os.path.join(RAW_DIR, 'Deepfix.json')

CORPUS_OUT_PATH      = os.path.join(CORPUS_OUT_DIR, 'rag_data.jsonl')
BLACK_QUERY_SET_PATH = os.path.join(QUERY_OUT_DIR, 'black', 'query_set.jsonl') # 黑盒 = 共享 + 独享
WHITE_QUERY_SET_PATH = os.path.join(QUERY_OUT_DIR, 'white', 'query_set.jsonl') # 白盒 = 共享 + 语料库重叠

# =========================================================
# 工具函数
# =========================================================

def extract_fields(record: dict, source: str) -> dict:
    """提取统一字段并标注来源（不含 id，由各步骤单独分配）"""
    return {
        'language': record['language'],
        'buggy_code': record['buggy_code'],
        'fixed_code': record['fixed_code'],
        'source': source,
    }

def assign_ids(data: list[dict]) -> list[dict]:
    """为列表中每条数据分配从 1 开始的自增整数 id"""
    return [{'id': i} | item for i, item in enumerate(data, 1)]

def write_jsonl(path: str, data: list[dict]):
    """将列表写入 JSONL 文件"""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  ✅ 已写入 {len(data)} 条数据 → {path}")

def format_bfp_dataset(dataset: list[dict], label: str = "") -> list[dict]:
    """
    对数据集中每条记录的 buggy_code 和 fixed_code 进行批量并行格式化。
    buggy_code 与 fixed_code 合并为一个大批次统一处理，减少进程池启动次数。
    格式化失败时保留原始代码（on_error='keep'）。
    """
    from src.utils.formatter import auto_format_c_code_batch
    n = len(dataset)
    tag = f"[{label}] " if label else ""
    buggy_codes = [item['buggy_code'] for item in dataset]
    fixed_codes  = [item['fixed_code']  for item in dataset]
    all_codes = buggy_codes + fixed_codes
    print(f"  {tag}格式化 buggy_code + fixed_code，共 {len(all_codes)} 条...")
    all_results = auto_format_c_code_batch(all_codes, on_error="keep")
    buggy_fmt = [r[0] for r in all_results[:n]]
    fixed_fmt  = [r[0] for r in all_results[n:]]
    return [
        {**item, 'buggy_code': b, 'fixed_code': f}
        for item, b, f in zip(dataset, buggy_fmt, fixed_fmt)
    ]

print(f"项目根目录: {PROJECT_ROOT}")

## 2 数据加载、洗牌与比例计算

In [ ]:
random.seed(SEED)

# 读取 CoCoNut（JSONL 格式，逐行解析）
coconut_data = []
with open(COCONUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            coconut_data.append(json.loads(line))
print(f"CoCoNut:   {len(coconut_data)} 条")

# 读取 Codeflaws（JSON 数组）
with open(CODEFLAWS_PATH, 'r', encoding='utf-8') as f:
    codeflaws_data = json.load(f)
print(f"Codeflaws: {len(codeflaws_data)} 条")

# 读取 DeepFix（JSON 数组）
with open(DEEPFIX_PATH, 'r', encoding='utf-8') as f:
    deepfix_data = json.load(f)
print(f"DeepFix:   {len(deepfix_data)} 条")

# 洗牌打乱
random.shuffle(coconut_data)
random.shuffle(codeflaws_data)
random.shuffle(deepfix_data)

# =========================================================
# 根据实际数据规模动态计算比例与分配量
# =========================================================
TOTAL_RAW = len(coconut_data) + len(codeflaws_data) + len(deepfix_data)
RATIO_COCONUT   = len(coconut_data)   / TOTAL_RAW
RATIO_CODEFLAWS = len(codeflaws_data) / TOTAL_RAW
RATIO_DEEPFIX   = len(deepfix_data)   / TOTAL_RAW

# --- 语料库各来源分配量 ---
CORPUS_COCONUT   = round(CORPUS_SIZE * RATIO_COCONUT)
CORPUS_CODEFLAWS = round(CORPUS_SIZE * RATIO_CODEFLAWS)
CORPUS_DEEPFIX   = CORPUS_SIZE - CORPUS_COCONUT - CORPUS_CODEFLAWS

# --- 查询集：共享 / 重叠 ---
# SHARED: 与语料库不相交，同时出现在白盒和黑盒中
# OVERLAP: 仅白盒从语料库中取；黑盒从另一批剩余数据中取
QUERY_SHARED_SIZE  = round(QUERY_SIZE * (1 - OVERLAP_RATE)) # 白盒与黑盒的共享部分
QUERY_OVERLAP_SIZE = QUERY_SIZE - QUERY_SHARED_SIZE # 白盒设置

# 共享部分按比例分配
QUERY_SHARED_COCONUT   = round(QUERY_SHARED_SIZE * RATIO_COCONUT)
QUERY_SHARED_CODEFLAWS = round(QUERY_SHARED_SIZE * RATIO_CODEFLAWS)
QUERY_SHARED_DEEPFIX   = QUERY_SHARED_SIZE - QUERY_SHARED_COCONUT - QUERY_SHARED_CODEFLAWS

# 黑盒独占部分（替代重叠的那部分）按比例分配
QUERY_BLACK_UNIQUE_COCONUT   = round(QUERY_OVERLAP_SIZE * RATIO_COCONUT)
QUERY_BLACK_UNIQUE_CODEFLAWS = round(QUERY_OVERLAP_SIZE * RATIO_CODEFLAWS)
QUERY_BLACK_UNIQUE_DEEPFIX   = QUERY_OVERLAP_SIZE - QUERY_BLACK_UNIQUE_COCONUT - QUERY_BLACK_UNIQUE_CODEFLAWS

# =========================================================
# 充分性检查
# =========================================================
needed_coconut   = CORPUS_COCONUT + QUERY_SHARED_COCONUT + QUERY_BLACK_UNIQUE_COCONUT
needed_codeflaws = CORPUS_CODEFLAWS + QUERY_SHARED_CODEFLAWS + QUERY_BLACK_UNIQUE_CODEFLAWS
needed_deepfix   = CORPUS_DEEPFIX + QUERY_SHARED_DEEPFIX + QUERY_BLACK_UNIQUE_DEEPFIX
assert len(coconut_data)   >= needed_coconut,   f"CoCoNut 不足: 需 {needed_coconut}, 有 {len(coconut_data)}"
assert len(codeflaws_data) >= needed_codeflaws, f"Codeflaws 不足: 需 {needed_codeflaws}, 有 {len(codeflaws_data)}"
assert len(deepfix_data)   >= needed_deepfix,   f"DeepFix 不足: 需 {needed_deepfix}, 有 {len(deepfix_data)}"

# =========================================================
# 打印配置摘要
# =========================================================
print(f"\n{'=' * 60}")
print("数据集划分配置（基于实际数据动态计算）")
print(f"{'=' * 60}")
print(f"原始数据总计:     {TOTAL_RAW}")
print(f"  CoCoNut:        {len(coconut_data):>6}  ({RATIO_COCONUT:.4%})")
print(f"  Codeflaws:      {len(codeflaws_data):>6}  ({RATIO_CODEFLAWS:.4%})")
print(f"  DeepFix:        {len(deepfix_data):>6}  ({RATIO_DEEPFIX:.4%})")
print(f"语料库规模:       {CORPUS_SIZE}")
print(f"  CoCoNut:        {CORPUS_COCONUT}")
print(f"  Codeflaws:      {CORPUS_CODEFLAWS}")
print(f"  DeepFix:        {CORPUS_DEEPFIX}")
print(f"查询集规模:       {QUERY_SIZE}  (共享 {QUERY_SHARED_SIZE} + 差异 {QUERY_OVERLAP_SIZE})")
print(f"  共享部分:       CoCoNut={QUERY_SHARED_COCONUT}, Codeflaws={QUERY_SHARED_CODEFLAWS}, DeepFix={QUERY_SHARED_DEEPFIX}")
print(f"  黑盒独占:       CoCoNut={QUERY_BLACK_UNIQUE_COCONUT}, Codeflaws={QUERY_BLACK_UNIQUE_CODEFLAWS}, DeepFix={QUERY_BLACK_UNIQUE_DEEPFIX}")
print(f"  白盒独占:       从语料库中抽取 {QUERY_OVERLAP_SIZE} 条")
print(f"重叠率 θ:         {OVERLAP_RATE}")
print(f"随机种子:         {SEED}")
print(f"{'=' * 60}")

## 3 构建语料数据集

In [ ]:
# 按比例从各数据集中抽取前 N 条作为语料库
corpus_coconut   = coconut_data[:CORPUS_COCONUT]
corpus_codeflaws = codeflaws_data[:CORPUS_CODEFLAWS]
corpus_deepfix   = deepfix_data[:CORPUS_DEEPFIX]

print(f"语料库子集: CoCoNut={len(corpus_coconut)}, Codeflaws={len(corpus_codeflaws)}, DeepFix={len(corpus_deepfix)}")

# 合并并统一字段，分配从 1 开始的连续 id
corpus_dataset = assign_ids(
    [extract_fields(r, 'CoCoNut')   for r in corpus_coconut] +
    [extract_fields(r, 'Codeflaws') for r in corpus_codeflaws] +
    [extract_fields(r, 'DeepFix')   for r in corpus_deepfix]
)

print(f"语料库总计: {len(corpus_dataset)} 条，id 范围: {corpus_dataset[0]['id']} ~ {corpus_dataset[-1]['id']}")
assert len(corpus_dataset) == CORPUS_SIZE, f"期望 {CORPUS_SIZE}，实际 {len(corpus_dataset)}"

corpus_dataset = format_bfp_dataset(corpus_dataset, label="corpus")
write_jsonl(CORPUS_OUT_PATH, corpus_dataset)

## 4 构建白盒查询集和黑盒查询集

构建逻辑：
1. **共享部分**（`(1 − θ) × QUERY_SIZE` 条）：从与语料库不相交的剩余数据中抽取，同时放入白盒和黑盒查询集。
2. **白盒独占部分**（`θ × QUERY_SIZE` 条）：从语料库中随机抽取，仅放入白盒查询集。
3. **黑盒独占部分**（`θ × QUERY_SIZE` 条）：从与语料库及共享部分均不相交的剩余数据中抽取，仅放入黑盒查询集。

In [ ]:
# =========================================================
# 4.1 共享部分：从语料库之后的剩余数据中按比例抽取
# =========================================================
remain_coconut   = coconut_data[CORPUS_COCONUT:]
remain_codeflaws = codeflaws_data[CORPUS_CODEFLAWS:]
remain_deepfix   = deepfix_data[CORPUS_DEEPFIX:]
print(f"剩余数据（语料库之后）: CoCoNut={len(remain_coconut)}, "
      f"Codeflaws={len(remain_codeflaws)}, DeepFix={len(remain_deepfix)}")

shared_coconut   = remain_coconut[:QUERY_SHARED_COCONUT]
shared_codeflaws = remain_codeflaws[:QUERY_SHARED_CODEFLAWS]
shared_deepfix   = remain_deepfix[:QUERY_SHARED_DEEPFIX]

shared_part = (
    [extract_fields(r, 'CoCoNut')   for r in shared_coconut] +
    [extract_fields(r, 'Codeflaws') for r in shared_codeflaws] +
    [extract_fields(r, 'DeepFix')   for r in shared_deepfix]
)
print(f"共享部分: {len(shared_part)} 条 "
      f"(CoCoNut={len(shared_coconut)}, Codeflaws={len(shared_codeflaws)}, DeepFix={len(shared_deepfix)})")

# =========================================================
# 4.2 白盒独占部分：从语料库中随机抽取
# =========================================================
corpus_all_raw = corpus_coconut + corpus_codeflaws + corpus_deepfix
random.shuffle(corpus_all_raw)
white_overlap_raw = corpus_all_raw[:QUERY_OVERLAP_SIZE]

def infer_source(record: dict) -> str:
    """根据字段特征推断数据来源"""
    if 'user_id' in record:
        return 'DeepFix'
    if 'bug_type' in record:
        return 'Codeflaws'
    return 'CoCoNut'

white_overlap_part = [extract_fields(r, infer_source(r)) for r in white_overlap_raw]
print(f"白盒独占（来自语料库）: {len(white_overlap_part)} 条")

# =========================================================
# 4.3 黑盒独占部分：从共享部分之后的剩余数据中按比例抽取
# =========================================================
remain2_coconut   = remain_coconut[QUERY_SHARED_COCONUT:]
remain2_codeflaws = remain_codeflaws[QUERY_SHARED_CODEFLAWS:]
remain2_deepfix   = remain_deepfix[QUERY_SHARED_DEEPFIX:]
print(f"剩余数据（共享之后）: CoCoNut={len(remain2_coconut)}, "
      f"Codeflaws={len(remain2_codeflaws)}, DeepFix={len(remain2_deepfix)}")

black_unique_coconut   = remain2_coconut[:QUERY_BLACK_UNIQUE_COCONUT]
black_unique_codeflaws = remain2_codeflaws[:QUERY_BLACK_UNIQUE_CODEFLAWS]
black_unique_deepfix   = remain2_deepfix[:QUERY_BLACK_UNIQUE_DEEPFIX]

black_unique_part = (
    [extract_fields(r, 'CoCoNut')   for r in black_unique_coconut] +
    [extract_fields(r, 'Codeflaws') for r in black_unique_codeflaws] +
    [extract_fields(r, 'DeepFix')   for r in black_unique_deepfix]
)
print(f"黑盒独占: {len(black_unique_part)} 条 "
      f"(CoCoNut={len(black_unique_coconut)}, Codeflaws={len(black_unique_codeflaws)}, DeepFix={len(black_unique_deepfix)})")

# =========================================================
# 4.4 组装白盒 / 黑盒查询集并写入
# =========================================================
white_dataset = assign_ids(shared_part + white_overlap_part)
black_dataset = assign_ids(shared_part + black_unique_part)

print(f"\n白盒查询集: {len(white_dataset)} 条  "
      f"(共享 {len(shared_part)} + 语料库重叠 {len(white_overlap_part)})")
print(f"黑盒查询集: {len(black_dataset)} 条  "
      f"(共享 {len(shared_part)} + 独占 {len(black_unique_part)})")

assert len(white_dataset) == QUERY_SIZE, f"白盒: 期望 {QUERY_SIZE}，实际 {len(white_dataset)}"
assert len(black_dataset) == QUERY_SIZE, f"黑盒: 期望 {QUERY_SIZE}，实际 {len(black_dataset)}"

white_dataset = format_bfp_dataset(white_dataset, label="white query")
black_dataset = format_bfp_dataset(black_dataset, label="black query")

write_jsonl(WHITE_QUERY_SET_PATH, white_dataset)
write_jsonl(BLACK_QUERY_SET_PATH, black_dataset)

# =========================================================
# 最终汇总
# =========================================================
print(f"\n{'=' * 60}")
print("数据集划分完成")
print(f"{'=' * 60}")
print(f"语料库:        {CORPUS_SIZE:>6} 条 → {CORPUS_OUT_PATH}")
print(f"白盒查询集:    {QUERY_SIZE:>6} 条 → {WHITE_QUERY_SET_PATH}")
print(f"黑盒查询集:    {QUERY_SIZE:>6} 条 → {BLACK_QUERY_SET_PATH}")
print(f"白盒与语料库重叠: {QUERY_OVERLAP_SIZE} 条 ({OVERLAP_RATE:.0%})")
print(f"黑盒与语料库重叠: 0 条 (0%)")
print(f"{'=' * 60}")